In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
import zipfile
from datasets import load_dataset, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModel,
    PreTrainedModel,
    PretrainedConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from dataclasses import dataclass
from typing import Any, Dict, List
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
MODEL_NAME = "answerdotai/ModernBERT-large"
MAX_LENGTH = 3000
N_FOLDS = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Clarity labels
clarity_label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
clarity_id2label = {v: k for k, v in clarity_label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

# Evasion labels
evasion_label2id = {
    "Explicit": 0,
    "Implicit": 1,
    "Dodging": 2,
    "General": 3,
    "Deflection": 4,
    "Partial/half-answer": 5,
    "Declining to answer": 6,
    "Claims ignorance": 7,
    "Clarification": 8
}
evasion_id2label = {v: k for k, v in evasion_label2id.items()}
evasion_labels = list(evasion_label2id.keys())

In [ ]:
# Load Data - only train dataset
ds = load_dataset("ailsntua/QEvasion")

train_full_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

# Handle empty evasion labels
train_full_df['evasion_label'] = train_full_df['evasion_label'].replace('', 'Explicit')
test_df['evasion_label'] = test_df['evasion_label'].replace('', 'Explicit')

print(f"Train samples: {len(train_full_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nClarity label distribution (train):")
print(train_full_df['clarity_label'].value_counts())
print(f"\nEvasion label distribution (train):")
print(train_full_df['evasion_label'].value_counts())

In [ ]:
def tokenize_function(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    
    tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    
    evasion_ids = []
    for l in examples["evasion_label"]:
        if l == '' or l is None:
            evasion_ids.append(0)  # Default to Explicit
        else:
            evasion_ids.append(evasion_label2id[l])
    tokenized["evasion_labels"] = evasion_ids
    
    return tokenized

# Tokenize entire datasets ONCE for efficiency
print("Tokenizing training dataset...")
train_full_dataset = Dataset.from_pandas(train_full_df, preserve_index=False)
train_tokenized_full = train_full_dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=train_full_dataset.column_names
)
print(f"✓ Tokenized {len(train_tokenized_full)} training samples\n")

print("Tokenizing test dataset...")
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_tokenized = test_dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=test_dataset.column_names
)
print(f"✓ Tokenized {len(test_tokenized)} test samples")

In [ ]:
@dataclass
class MultiTaskDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        clarity_labels = torch.tensor([f["clarity_labels"] for f in features], dtype=torch.long)
        evasion_labels = torch.tensor([f["evasion_labels"] for f in features], dtype=torch.long)
        
        features_clean = [{k: v for k, v in f.items() if k not in ["clarity_labels", "evasion_labels"]} for f in features]
        
        batch = self.tokenizer.pad(
            features_clean,
            padding=True,
            return_tensors="pt"
        )
        
        batch["clarity_labels"] = clarity_labels
        batch["evasion_labels"] = evasion_labels
        
        return batch

data_collator = MultiTaskDataCollator(tokenizer=tokenizer)

In [ ]:
class ModernBERTMultiHeadConfig(PretrainedConfig):
    model_type = "modernbert_multihead"
    
    def __init__(
        self,
        base_model_name: str = "answerdotai/ModernBERT-large",
        num_clarity_labels: int = 3,
        num_evasion_labels: int = 9,
        hidden_size: int = 1024,
        hidden_dropout_prob: float = 0.1,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.base_model_name = base_model_name
        self.num_clarity_labels = num_clarity_labels
        self.num_evasion_labels = num_evasion_labels
        self.hidden_size = hidden_size
        self.hidden_dropout_prob = hidden_dropout_prob


class ModernBERTMultiHead(PreTrainedModel):
    config_class = ModernBERTMultiHeadConfig
    supports_gradient_checkpointing = True
    
    def __init__(self, config: ModernBERTMultiHeadConfig):
        super().__init__(config)
        self.config = config
        
        # Load the base encoder
        self.encoder = AutoModel.from_pretrained(
            config.base_model_name,
            reference_compile=False, 
            attn_implementation="sdpa",
            trust_remote_code=True
        )
        
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        
        # Separate classification heads
        self.clarity_classifier = nn.Linear(config.hidden_size, config.num_clarity_labels)
        self.evasion_classifier = nn.Linear(config.hidden_size, config.num_evasion_labels)
        
        # Initialize weights
        self._init_weights(self.clarity_classifier)
        self._init_weights(self.evasion_classifier)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if module.bias is not None:
                module.bias.data.zero_()

    def _set_gradient_checkpointing(self, module, value=False):
        if isinstance(module, (ModernBERTMultiHead,)):
            module.encoder.gradient_checkpointing = value
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        clarity_labels=None,
        evasion_labels=None,
        return_dict=True,
        **kwargs
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Use CLS token embedding
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        cls_embedding = self.dropout(cls_embedding)
        
        # Two separate classification heads
        logits_clarity = self.clarity_classifier(cls_embedding)
        logits_evasion = self.evasion_classifier(cls_embedding)
        
        loss = None
        if clarity_labels is not None and evasion_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss_clarity = loss_fct(logits_clarity, clarity_labels)
            loss_evasion = loss_fct(logits_evasion, evasion_labels)
            loss = loss_clarity + loss_evasion
        
        return {
            "loss": loss,
            "logits": logits_clarity,  # Default logits for Trainer
            "logits_clarity": logits_clarity,
            "logits_evasion": logits_evasion,
        }


def create_model():
    config = ModernBERTMultiHeadConfig(
        base_model_name=MODEL_NAME,
        num_clarity_labels=3,
        num_evasion_labels=9,
        hidden_size=1024,
        hidden_dropout_prob=0.1
    )
    return ModernBERTMultiHead(config)

In [ ]:
def compute_metrics(eval_pred):
    """Compute metrics based on clarity predictions (primary task)"""
    logits, labels = eval_pred
    # labels here is the clarity_labels (first label in label_names)
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    return {'accuracy': accuracy, 'f1': f1}


class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
def predict_multihead(model, dataset, data_collator, batch_size=8):
    """Run prediction and return both clarity and evasion probabilities"""
    from torch.utils.data import DataLoader
    
    model.eval()
    model.to(device)
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=data_collator,
        shuffle=False
    )
    
    all_clarity_logits = []
    all_evasion_logits = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            all_clarity_logits.append(outputs['logits_clarity'].cpu())
            all_evasion_logits.append(outputs['logits_evasion'].cpu())
    
    clarity_logits = torch.cat(all_clarity_logits, dim=0)
    evasion_logits = torch.cat(all_evasion_logits, dim=0)
    
    clarity_probs = torch.softmax(clarity_logits, dim=-1).numpy()
    evasion_probs = torch.softmax(evasion_logits, dim=-1).numpy()
    
    return clarity_probs, evasion_probs

In [ ]:
# 5-Fold Cross Validation with Multi-Head Model
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage for results
fold_test_clarity_probs = []
fold_test_evasion_probs = []
fold_oof_clarity_probs = []
fold_oof_evasion_probs = []
fold_oof_indices = []
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_full_df, train_full_df['clarity_label']), 1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")
    print(f"Train: {len(train_idx)} samples, Val: {len(val_idx)} samples")
    
    train_dataset = train_tokenized_full.select(train_idx)
    val_dataset = train_tokenized_full.select(val_idx)
    
    # Create fresh model for each fold
    model = create_model()
    
    training_args = TrainingArguments(
        output_dir=f"./results_modernbert_multihead_fold{fold}",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=10,
        warmup_steps=100,
        weight_decay=0.10,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        bf16=True,
        bf16_full_eval=True,
        optim="adamw_torch_fused",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        seed=SEED + fold,
        report_to="none",
        label_names=["clarity_labels", "evasion_labels"],
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1")],
    )
    
    trainer.train()
    
    # Evaluate on validation set
    val_results = trainer.evaluate()
    fold_metrics.append({
        'fold': fold,
        'val_accuracy': val_results['eval_accuracy'],
        'val_f1': val_results['eval_f1']
    })
    
    print(f"\nFold {fold} Result: Accuracy={val_results['eval_accuracy']:.4f}, F1={val_results['eval_f1']:.4f}")
    
    # Get OOF predictions (validation set)
    oof_clarity_probs, oof_evasion_probs = predict_multihead(model, val_dataset, data_collator)
    fold_oof_clarity_probs.append(oof_clarity_probs)
    fold_oof_evasion_probs.append(oof_evasion_probs)
    fold_oof_indices.append(val_idx)
    
    # Get test predictions
    test_clarity_probs, test_evasion_probs = predict_multihead(model, test_tokenized, data_collator)
    fold_test_clarity_probs.append(test_clarity_probs)
    fold_test_evasion_probs.append(test_evasion_probs)
    
    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")

In [ ]:
# Fold-level Metrics Summary
print("\n" + "="*60)
print("FOLD-LEVEL METRICS")
print("="*60)

metrics_df = pd.DataFrame(fold_metrics)
print(metrics_df.to_string(index=False))

print(f"\nAverage Validation Accuracy: {metrics_df['val_accuracy'].mean():.4f} ± {metrics_df['val_accuracy'].std():.4f}")
print(f"Average Validation Macro F1: {metrics_df['val_f1'].mean():.4f} ± {metrics_df['val_f1'].std():.4f}")

In [ ]:
# OOF Analysis - CLARITY
print("\n" + "="*60)
print("OOF ANALYSIS - CLARITY TASK")
print("="*60)

# Reconstruct full OOF predictions
oof_clarity_probs_full = np.zeros((len(train_full_df), 3))
for probs, idxs in zip(fold_oof_clarity_probs, fold_oof_indices):
    oof_clarity_probs_full[idxs] = probs

oof_clarity_preds = np.argmax(oof_clarity_probs_full, axis=1)
oof_clarity_labels = [clarity_id2label[p] for p in oof_clarity_preds]
y_true_clarity = train_full_df['clarity_label'].tolist()

print("\nClassification Report (OOF - Clarity):")
print(classification_report(y_true_clarity, oof_clarity_labels, labels=clarity_labels, zero_division=0))

print("Confusion Matrix (OOF - Clarity):")
cm_clarity = confusion_matrix(y_true_clarity, oof_clarity_labels, labels=clarity_labels)
print(f"Labels: {clarity_labels}")
print(cm_clarity)

oof_clarity_acc = accuracy_score(y_true_clarity, oof_clarity_labels)
oof_clarity_f1 = f1_score(y_true_clarity, oof_clarity_labels, average='macro', zero_division=0)
print(f"\nOOF Clarity Accuracy: {oof_clarity_acc:.4f}")
print(f"OOF Clarity Macro F1: {oof_clarity_f1:.4f}")

In [ ]:
# OOF Analysis - EVASION
print("\n" + "="*60)
print("OOF ANALYSIS - EVASION TASK")
print("="*60)

# Reconstruct full OOF predictions
oof_evasion_probs_full = np.zeros((len(train_full_df), 9))
for probs, idxs in zip(fold_oof_evasion_probs, fold_oof_indices):
    oof_evasion_probs_full[idxs] = probs

oof_evasion_preds = np.argmax(oof_evasion_probs_full, axis=1)
oof_evasion_labels = [evasion_id2label[p] for p in oof_evasion_preds]
y_true_evasion = train_full_df['evasion_label'].tolist()

print("\nClassification Report (OOF - Evasion):")
print(classification_report(y_true_evasion, oof_evasion_labels, labels=evasion_labels, zero_division=0))

print("Confusion Matrix (OOF - Evasion):")
cm_evasion = confusion_matrix(y_true_evasion, oof_evasion_labels, labels=evasion_labels)
print(f"Labels: {evasion_labels}")
print(cm_evasion)

oof_evasion_acc = accuracy_score(y_true_evasion, oof_evasion_labels)
oof_evasion_f1 = f1_score(y_true_evasion, oof_evasion_labels, average='macro', zero_division=0)
print(f"\nOOF Evasion Accuracy: {oof_evasion_acc:.4f}")
print(f"OOF Evasion Macro F1: {oof_evasion_f1:.4f}")

In [ ]:
# Ensemble Test Results - CLARITY
print("\n" + "="*60)
print("ENSEMBLE TEST RESULTS - CLARITY TASK")
print("="*60)

ensemble_clarity_probs = np.mean(fold_test_clarity_probs, axis=0)
ensemble_clarity_preds = np.argmax(ensemble_clarity_probs, axis=1)
ensemble_clarity_labels = [clarity_id2label[p] for p in ensemble_clarity_preds]

y_true_test_clarity = test_df['clarity_label'].tolist()

print(f"\nAccuracy: {accuracy_score(y_true_test_clarity, ensemble_clarity_labels):.4f}")
print(f"Macro F1: {f1_score(y_true_test_clarity, ensemble_clarity_labels, average='macro', labels=clarity_labels, zero_division=0):.4f}")

print("\nClassification Report (Test Ensemble - Clarity):")
print(classification_report(y_true_test_clarity, ensemble_clarity_labels, labels=clarity_labels, zero_division=0))

print("Confusion Matrix (Test - Clarity):")
cm_test_clarity = confusion_matrix(y_true_test_clarity, ensemble_clarity_labels, labels=clarity_labels)
print(f"Labels: {clarity_labels}")
print(cm_test_clarity)

In [ ]:
# Ensemble Test Results - EVASION
print("\n" + "="*60)
print("ENSEMBLE TEST RESULTS - EVASION TASK")
print("="*60)

ensemble_evasion_probs = np.mean(fold_test_evasion_probs, axis=0)
ensemble_evasion_preds = np.argmax(ensemble_evasion_probs, axis=1)
ensemble_evasion_labels = [evasion_id2label[p] for p in ensemble_evasion_preds]

y_true_test_evasion = test_df['evasion_label'].tolist()

print(f"\nAccuracy: {accuracy_score(y_true_test_evasion, ensemble_evasion_labels):.4f}")
print(f"Macro F1: {f1_score(y_true_test_evasion, ensemble_evasion_labels, average='macro', labels=evasion_labels, zero_division=0):.4f}")

print("\nClassification Report (Test Ensemble - Evasion):")
print(classification_report(y_true_test_evasion, ensemble_evasion_labels, labels=evasion_labels, zero_division=0))

print("Confusion Matrix (Test - Evasion):")
cm_test_evasion = confusion_matrix(y_true_test_evasion, ensemble_evasion_labels, labels=evasion_labels)
print(f"Labels: {evasion_labels}")
print(cm_test_evasion)

In [ ]:
# Save predictions
os.makedirs("./predictions", exist_ok=True)

# Save OOF probabilities
np.save("predictions/oof_clarity_probs_multihead.npy", oof_clarity_probs_full)
np.save("predictions/oof_evasion_probs_multihead.npy", oof_evasion_probs_full)

# Save test ensemble probabilities
np.save("predictions/test_clarity_probs_multihead.npy", ensemble_clarity_probs)
np.save("predictions/test_evasion_probs_multihead.npy", ensemble_evasion_probs)

print("✓ Saved OOF and test probabilities for both tasks")

# Save clarity predictions for submission
prediction_filename = "./predictions/prediction_multihead"
with open(prediction_filename, 'w') as f:
    for label in ensemble_clarity_labels:
        f.write(f"{label}\n")

zip_filename = "./predictions/prediction_multihead.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(prediction_filename, arcname="prediction")

print(f"✓ Saved clarity predictions to {zip_filename}")

In [ ]:
# Final Summary
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print("\n--- Cross-Validation Results ---")
print(f"Validation Accuracy: {metrics_df['val_accuracy'].mean():.4f} ± {metrics_df['val_accuracy'].std():.4f}")
print(f"Validation Macro F1: {metrics_df['val_f1'].mean():.4f} ± {metrics_df['val_f1'].std():.4f}")

print("\n--- OOF Results ---")
print(f"Clarity - Accuracy: {oof_clarity_acc:.4f}, Macro F1: {oof_clarity_f1:.4f}")
print(f"Evasion - Accuracy: {oof_evasion_acc:.4f}, Macro F1: {oof_evasion_f1:.4f}")

print("\n--- Test Ensemble Results ---")
test_clarity_acc = accuracy_score(y_true_test_clarity, ensemble_clarity_labels)
test_clarity_f1 = f1_score(y_true_test_clarity, ensemble_clarity_labels, average='macro', zero_division=0)
test_evasion_acc = accuracy_score(y_true_test_evasion, ensemble_evasion_labels)
test_evasion_f1 = f1_score(y_true_test_evasion, ensemble_evasion_labels, average='macro', zero_division=0)

print(f"Clarity - Accuracy: {test_clarity_acc:.4f}, Macro F1: {test_clarity_f1:.4f}")
print(f"Evasion - Accuracy: {test_evasion_acc:.4f}, Macro F1: {test_evasion_f1:.4f}")